# **Dynammic Programming**

<hr>

### **Part 2**: Reinforcement Learning from Zero to One

*African Institute for Mathematical Sciences (AIMS), South Africa
3 March, 2026*

**Arnu Pretorius** - Head of Decision-Making AI Research, InstaDeep

<!--

Flow

* N=10, deterministic
* N=10, stochastic (hint towards the concept of exploration)

-->



## **The Numberline**



In [ ]:
# @title Setup
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.animation as animation

class NumberLineEnv:
    def __init__(self, num_states=10, max_steps=50, move_probability=0.95, reward_type='sparse'):
        self.num_states = num_states
        self.max_steps = max_steps
        self.reward_state = self.num_states-1
        self.move_probability = move_probability  # Probability of moving in the intended direction
        self.reward_type = reward_type

    def reset(self, key):
        """Initialize the environment state."""
        state = (0, 0, key)
        timestep = (0, 0, False)
        return state, timestep

    def step(self, state, action):
        """Takes an action in the environment."""
        obs, step_count, key = state
        new_key, sub_key = jax.random.split(key)

        # Determine movement direction (intended or opposite)
        intended_direction = jnp.where(jax.random.uniform(sub_key) < self.move_probability, action, -1)

        # Update position based on stochastic action
        next_obs = jnp.clip(obs + intended_direction, 0, self.reward_state)

        # Calculate instantaneous reward
        is_at_goal = next_obs == self.reward_state
        reward = jax.lax.cond(
            self.reward_type == 'sparse',
            lambda x: jnp.where(x, 1, 0),
            lambda x: jnp.where(~x, -1, 0),
            is_at_goal,
        )
        done = (jnp.where(step_count == self.max_steps-1, 1, 0) + is_at_goal) > 0

        # Update state
        state = (next_obs, step_count + 1, new_key)
        timestep = (next_obs, reward, done)

        return state, timestep

    def _prepare_figure(self, ax):

        # Add labels and legend
        ax.set_xticks([])
        ax.set_yticks([])

    def _draw_state(self, ax, state):
        obs, _, _ = state

        # Create a line for the state space
        states = range(self.num_states)
        ax.plot(states, [0] * len(states), 'o-', color='gray', markersize=10)

        # Mark agent and target state
        ax.plot(self.num_states-1, 0)
        color = 'green' if obs == self.num_states - 1 else 'blue'
        ax.plot(obs, 0, 'o', color=color, markersize=15, label='Agent')

    def render(self, state):
        # Prepare figure and axis
        fig, ax = plt.subplots()
        ax.clear()
        self._prepare_figure(ax)

        # Mark the current state of the agent
        self._draw_state(ax, state)
        plt.show()

    def animate(self, states, interval=200, save_path=None):
        fig = plt.figure()
        fig.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0, hspace=0)
        ax = fig.add_subplot(111)
        plt.close(fig)
        self._prepare_figure(ax)

        def make_frame(state) -> None:
            ax.clear()
            self._prepare_figure(ax)
            self._draw_state(ax, state)

        # Create the animation object.
        self._animation = animation.FuncAnimation(
            fig,
            make_frame,
            frames=states,
            interval=interval,
        )

        # Save the animation as a gif.
        if save_path:
            self._animation.save(save_path)

        return self._animation

# policy helper functions
def make_deterministic_policy(fixed_key, probs=None):
    def policy(x, key=None):
        return jax.random.choice(fixed_key, jnp.array([-1, 1]), shape=(num_states,))[x]
    return policy

def make_stochastic_policy(burn_key, probs=None):
    if probs is None:
        probs = jnp.array([0.5, 0.5])
    def policy(x, key):
        key, sub_key = jax.random.split(key)
        return jax.random.choice(sub_key, jnp.array([-1, 1]), p=probs)
    return policy

def sample_policy(policy, key):
    keys = jax.random.split(key, num_states)
    return jnp.array([policy(x, key) for x, key in enumerate(keys)])

In [ ]:
# create environment
num_states = 10
num_actions = 2
env = NumberLineEnv(num_states=num_states)
reset_fn = jax.jit(env.reset)
step_fn = jax.jit(env.step)

# visualise environment
key = jax.random.PRNGKey(3)
env_state, timestep = reset_fn(key)
env.render(env_state)

In [ ]:
##########################################
# Change for different types of policies #
##########################################
# policy_fn = make_deterministic_policy
policy_fn = make_stochastic_policy

key, policy_key = jax.random.split(key)
policy = policy_fn(policy_key)


# **Policy Iteration**

## Policy Evaluation $←→$ Policy Improvement



MDP dynamics --- $\textbf{P}_{sas'} = p(s'|s,a)$

In [ ]:
# MDP dynamics p(s'|s,a)
move_probability = 0.95
gamma = 0.9
P = jnp.zeros((num_states, num_actions, num_states))

for s in range(num_states):
    for a in range(num_actions):
        action = 2*a-1
        next_s = jnp.clip(s+action, 0, num_states-1)
        move_left = jnp.clip(s-1, 0, num_states-1)
        if a == 0 and s != num_states - 1:
            P = P.at[(s, a, next_s)].set(1)
        elif s == num_states - 1:
            P = P.at[(s, a, s)].set(1)
        else:
            P = P.at[(s, a, next_s)].set(move_probability)
            P = P.at[(s, a, move_left)].set(1-move_probability)

print("Dynamics tensor shape: \n", P.shape)
print("\nDynamics tensor: \n", P)

Reward function --- $\textbf{R}_{sas'} = r(s, a, s')$

In [ ]:
R = jnp.zeros((num_states, num_actions, num_states))
reward_type = 'sparse'

if reward_type == 'sparse':
    reward_state = num_states-1
    R = R.at[(reward_state-1, 1, reward_state)].set(1)

if reward_type == 'dense':
    reward_state = num_states-1
    R = (R + 1)*(-1)
    R = R.at[(reward_state-1, 1, reward_state)].set(0)

print("Reward tensor shape: \n", R.shape)
print("\nReward tensor: \n", R)

## Policy Evaluation: $V → v_\pi$
   $$
   V(s) = \sum_{a \in A} \pi(a|s) \sum_{s' \in S} p(s'|s, a) \left[r + \gamma V(s')\right],
   $$
where $\mathbf{\pi}_{sa} = \pi(a|s) \in \mathbb{R}^{|S|\times|A|}$ and $V \in \mathbb{R}^{|S|}$.

In [ ]:
# Initial policy
if policy_fn == make_stochastic_policy:
    pi = jnp.full((num_states, num_actions), 1/num_actions)  # Uniform policy
else:
    fixed_policy = jnp.array(((sample_policy(policy, key) - 1)*-1)/2, dtype=jnp.int32)
    pi = jnp.ones((num_states, num_actions))

    # List of indices to set to one
    action_idx = [(i, j) for i, j in enumerate(fixed_policy)]

    # Update the matrix
    for idx in action_idx:
        pi = pi.at[idx].set(0)

print("Policy shape: \n", pi.shape)
print("\nPolicy: \n", pi)

In [ ]:
# Initial value function
V = jnp.zeros(num_states)

print("Value vector shape: \n", V.shape)
print("\nValue function: \n", V)

In [ ]:
def policy_evaluation(pi, P, R, V, gamma, threshold=1e-5):
    while True:
        V_new = jnp.sum(pi * jnp.sum(P * (R + gamma * V), axis=2), axis=1)
        if jnp.max(jnp.abs(V_new - V)) < threshold:
            break
        V = V_new
    return V_new

In [ ]:
# Perform policy evaluation
V_pi = policy_evaluation(pi, P, R, V, gamma)
V_pi

## Policy Improvement: $\pi → \text{greedy}(V)$

   $$
   \pi'(s) = \arg\max_{a \in A} \sum_{s' \in S} p(s' | s, a) \left[r + \gamma V^\pi(s')\right]
   $$


In [ ]:
def policy_improvement(pi, P, R, V, gamma):
    action_idx = jnp.argmax(jnp.sum(P * (R + gamma * V), axis=2), axis=1)
    pi = jnp.zeros_like(pi)
    for s, a_id in zip(range(num_states), action_idx):
        pi = pi.at[(s, a_id)].set(1)
    return pi

In [ ]:
# Perform policy improvement
pi_prime = policy_improvement(pi, P, R, V_pi, gamma)
pi_prime

## *Repeat*

In [ ]:
# Perform policy evaluation
V_pi = policy_evaluation(pi_prime, P, R, V_pi, gamma)
V_pi

In [ ]:
# Perform policy improvement
pi_prime = policy_improvement(pi_prime, P, R, V_pi, gamma)
pi_prime

## **Policy Iteration algorithm** ([*repeat*: evaluate $→$ improve] → $v_*, \pi_*$ )

In [ ]:
def policy_iteration(pi, P, R, V, gamma, threshold=1e-5):
    V_hist = []
    pi_hist = []
    V_hist.append(V)
    pi_hist.append(pi)

    while True:
        V_pi = policy_evaluation(pi, P, R, V, gamma)
        pi_prime = policy_improvement(pi, P, R, V_pi, gamma)

        V_hist.append(V_pi)
        pi_hist.append(pi_prime)

        if jnp.max(jnp.abs(pi_prime - pi)) < threshold:
            break
        V = V_pi
        pi = pi_prime
    return V_hist, pi_hist

In [ ]:
# Perform policy iteration
V_hist, pi_hist = policy_iteration(pi, P, R, V, gamma)
print("Optimal values: \n", V_hist[-1])
print("\nOptimal policy: \n", pi_hist[-1])

In [ ]:
# Create a figure with two subplots
plt.figure(figsize=(12, 4))  # Adjust the figure size as needed

# First subplot
plt.subplot(1, 2, 1)  # (1, 2, 1) means 1 row, 2 columns, first subplot
plt.imshow(jnp.array(V_hist), aspect='auto', cmap='viridis')
plt.colorbar()
plt.title("Change in values")
plt.xlabel("States")
plt.ylabel("Iteration")

# Second subplot
plt.subplot(1, 2, 2)  # (1, 2, 2) means 1 row, 2 columns, second subplot
plt.imshow(jnp.array(pi_hist)[:, :, 1], aspect='auto', cmap='viridis')
plt.colorbar()
plt.title("Change in policy")
plt.xlabel("States")
plt.ylabel("Iteration")

plt.show()

## **Value Iteration Algorithm** ([*repeat*: evaluate  +  improve] → $v_*, π_*$)

Bellman *optimality* equations

$$
V(s) = \max_{a \in A} \sum_{s' \in S} p(s'|s, a) [r + \gamma V(s')]
$$

$$
\pi^\prime(s) = \arg\max_{a \in A} \sum_{s' \in S} p(s'|s, a) [r + \gamma V(s')]
$$



In [ ]:
# Initial value function
V

In [ ]:
# Initial policy
pi

In [ ]:
# Value iteration using the Bellman optimality update
def value_iteration(pi, P, R, V, gamma, threshold=1e-5):
    while True:
        V_star = jnp.max(jnp.sum(P * (R + gamma * V), axis=2), axis=1)
        if jnp.max(jnp.abs(V_star - V)) < threshold:
            break
        V = V_star
    return V_star

In [ ]:
# Perform policy evaluation
V_star = value_iteration(pi, P, R, V, gamma)
V_star

In [ ]:
pi_star = policy_improvement(pi, P, R, V_star, gamma)
pi_star